# Semantic retrieval

**Semantic** means meaning. Semantic retrieval finds chunks that discuss the
same idea as a question, even when they use different words.

This tutorial introduces three ideas:

- **Indexing** — embed each repository chunk once and store its vector.
- **Retrieval** — compare a new question with those stored vectors.
- **Ranking** — return chunks with the highest cosine similarity first.

The notebook uses the public retrieval interface in
`qachat/retrievals/retrievals.py`. The semantic strategy itself lives in
`qachat/retrievals/strategies/retrieval_semantic.py`.

No chat model writes an answer yet. Tutorial 80 adds lexical retrieval, where
the actual words and software identifiers matter.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Load an embedding model

Semantic retrieval uses the same embedding layer as tutorial 60. The chat model used later for answering is independent from this model.


In [ ]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}


In [ ]:
from qachat.runtime import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")


In [ ]:
from qachat.embeddings import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare repository chunks

These are the same lower layers used in tutorials 40–60: select files, read
them, extract natural source units, and split only units that are too large.


In [ ]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py", ".md"],
    include_paths=["qachat", "README.md"],
)

print(f"Selected files: {len(report['selected_files'])}")


In [ ]:
from qachat.sources.sources import (
    extract_source_units,
    read_repository_files,
)

source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Read files: {len(source_files)}")
print(f"Natural units: {len(units)}")


In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 3. Build the semantic index

The index is prepared **before** a user searches.

Each repository chunk is embedded once:

`chunk → embedding → semantic index`

The stored vectors can then be reused for many questions.


In [ ]:
from qachat.retrievals.retrievals import build_semantic_index

semantic_index = build_semantic_index(
    chunks,
    embedding_session,
)

assert len(semantic_index.chunks) == len(chunks)
assert len(semantic_index.embeddings) == len(chunks)

print(f"Indexed chunks: {len(semantic_index.chunks)}")
print(f"Vector size: {len(semantic_index.embeddings[0])}")


In [ ]:
first_chunk = semantic_index.chunks[0]
first_vector = semantic_index.embeddings[0]

print(f"Chunk ID: {first_chunk.identifier}")
print(f"Source: {first_chunk.path}")
print(f"Stored vector: {first_vector}")


## 4. Search by meaning

A new question is embedded with the same model. `semantic_search` compares
that vector with every indexed chunk using cosine similarity and ranks the
closest meanings first.

The question deliberately avoids a function name, so this demonstrates
semantic rather than exact-word retrieval.


In [ ]:
from qachat.query_formats import (
    SOFTWARE_REPOSITORY_INSTRUCTION,
    format_qwen3_query,
)

QUESTION = "How does QAChat decide which computation backend to run on?"

query_text = format_qwen3_query(
    question=QUESTION,
    instruction=SOFTWARE_REPOSITORY_INSTRUCTION,
)

print(query_text)


In [ ]:
from qachat.retrievals.strategies.retrieval_semantic import semantic_search

results = semantic_search(
    semantic_index,
    query_text,
    embedding_session,
    count=5,
)


In [ ]:
scores = [result.score for result in results]

assert results
assert scores == sorted(scores, reverse=True)

print(f"Retrieved chunks: {len(results)}")
print("Highest score:", scores[0])


## 5. Inspect the ranking

Cosine similarity compares the direction of the question and chunk vectors. A higher score places a chunk earlier in the results. The score orders candidates; it is not the probability that a chunk is correct.


In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(f"{rank}. {result.score:.3f} | {location} | {name}")


In [ ]:
best = results[0]

print("Most relevant chunk:")
print()
print(best.chunk.text)


In [ ]:
for rank, result in enumerate(results):
    print(
        f"\n>>> Chunk {rank}. {result.score:.3f} | {location} | {name} <<<\n"
    )
    print(f"{result.chunk.text}")

## 6. What happened?

Indexing happens once:

`chunk → embedding → semantic index`

For each question:

`question → embedding → cosine similarity → ranking → top chunks`

Tutorial 80 uses a different index for lexical retrieval, where matching words
and software identifiers matter more than meaning.


## 7. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Result

QAChat embedded the repository chunks once and stored their vectors in a semantic index. It then embedded one question, ranked the chunks by cosine similarity, and returned the closest meanings. No LLM answer was generated.
